# Lab 5: Lexical Semantics

Find the most frequent WordNet synset of each (lemma, category) pair, and compare the noun and verb pairs with four WordNet similarities: Path, Leacock-Chodorow, Wu-Palmer and Lin.

## Setup

Import the libraries and download WordNet and the Brown information content (needed by Lin).

In [1]:
import sys
from pathlib import Path

# Colab: install packages, mount Google Drive and use the Colab Notebooks/ihlt folder as root.
# Local: notebooks live in labs/, shared resources in the project root.
# This is just some setup for file organization
if "google.colab" in sys.modules:
    %pip install -q nltk pandas
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/Colab Notebooks/ihlt")
else:
    ROOT = Path.cwd().parent

NLTK_DIR = ROOT / "nltk_data"

# Notebook libs
from itertools import combinations

import nltk
import pandas as pd

# NLTK packages needed by this notebook
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("wordnet", download_dir=NLTK_DIR)
nltk.download("omw-1.4", download_dir=NLTK_DIR)
nltk.download("wordnet_ic", download_dir=NLTK_DIR)

from nltk.corpus import wordnet as wn
from nltk.corpus import wordnet_ic

# Information content computed on the Brown corpus (needed by the Lin similarity).
brown_ic = wordnet_ic.ic("ic-brown.dat")

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package wordnet_ic to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package wordnet_ic is already up-to-date!


## Most frequent synset of each pair

Look up each unique (lemma, category) pair in WordNet and take its first sense, which is the most frequent one. The words without a synset in the given category are printed with the categories WordNet does have for them.

In [2]:
# The (lemma, category) pairs of the statement.
pairs = [
    ("the", "DT"), ("man", "NN"), ("swim", "VB"), ("with", "PR"), ("a", "DT"),
    ("girl", "NN"), ("and", "CC"), ("a", "DT"), ("boy", "NN"), ("whilst", "PR"),
    ("the", "DT"), ("woman", "NN"), ("walk", "VB"),
    ("the", "DT"), ("dog", "NN"), ("run", "VB"), ("after", "PR"), ("a", "DT"),
    ("cat", "NN"),
]

# Some pairs are repeated ("the", "a"), so keep each one once, in order of appearance.
unique_pairs = list(dict.fromkeys(pairs))

# WordNet only has nouns, verbs, adjectives and adverbs.
# Determiners (DT), prepositions (PR) and conjunctions (CC) have no WordNet category.
wn_pos = {"NN": wn.NOUN, "VB": wn.VERB, "JJ": wn.ADJ, "RB": wn.ADV}

# WordNet lists the senses of a lemma from the most to the least frequent
# so the most frequent synset is the first one. The counts are shown to check it.
synsets = {}
rows = []

def lemma_count(synset, lemma):
    return sum(l.count() for l in synset.lemmas() if l.name() == lemma)


for lemma, category in unique_pairs:
    pos = wn_pos.get(category)
    senses = wn.synsets(lemma, pos) if pos else []
    if not senses:
        # Some lemmas exist only in other categories (e.g. "a" as a noun).
        found = wn.synsets(lemma)
        categories = sorted({s.pos() for s in found})
        print(f"{lemma:<7} {category}: {len(found)} synsets in WordNet, categories {categories}")
        continue
    synsets[lemma] = senses[0]
    rows.append({"lemma": lemma, "category": category, "synset": senses[0].name(),
                 "count": lemma_count(senses[0], lemma),
                 "2nd sense": senses[1].name(), "2nd count": lemma_count(senses[1], lemma),
                 "definition": senses[0].definition()})

print(f"\nPairs: {len(pairs)}    unique pairs: {len(unique_pairs)}    Pairs available in WordNet with given categories: {len(synsets)}")

pd.set_option("display.max_colwidth", 70)
pd.DataFrame(rows).astype({"count": "Int64", "2nd count": "Int64"}).set_index(["lemma", "category"])

the     DT: 0 synsets in WordNet, categories []
with    PR: 0 synsets in WordNet, categories []
a       DT: 7 synsets in WordNet, categories ['n']
and     CC: 0 synsets in WordNet, categories []
whilst  PR: 0 synsets in WordNet, categories []
after   PR: 3 synsets in WordNet, categories ['r', 's']

Pairs: 19    unique pairs: 15    Pairs available in WordNet with given categories: 9


,,synset,count,2nd sense,2nd count,definition
lemma,category,,,,,
man,NN,man.n.01,749,serviceman.n.01,346,an adult person who is male (as opposed to a woman)
swim,VB,swim.v.01,12,float.v.02,1,travel through water
girl,NN,girl.n.01,80,female_child.n.01,57,a young woman
boy,NN,male_child.n.01,144,boy.n.02,42,a youthful male person
woman,NN,woman.n.01,143,woman.n.02,1,an adult female person (as opposed to a man)
walk,VB,walk.v.01,184,walk.v.02,3,use one's feet to advance; advance by steps
dog,NN,dog.n.01,42,frump.n.01,0,a member of the genus Canis (probably descended from the common wo...
run,VB,run.v.01,106,scat.v.01,38,"move fast by using one's feet, with one foot off the ground at any..."
cat,NN,cat.n.01,18,guy.n.01,0,feline mammal usually having thick soft fur and no ability to roar...


## Noun vs. verb

Try the four similarities on *man* (noun) and *swim* (verb). Their taxonomies have different roots, so there is no real common ancestor. The check with `simulate_root=False` shows that Path and Wu-Palmer only get a value through a fake root.

In [3]:
# Check for similarities comparison in different categories
man, swim = synsets["man"], synsets["swim"]

# The two taxonomies have different roots, so there is no real LCS.
# NLTK adds a fake root (*ROOT*) on top of both when a verb is involved.
print("Roots:", man.root_hypernyms(), swim.root_hypernyms())
print("LCS:", man.lowest_common_hypernyms(swim))
print("LCS with fake root:", man.lowest_common_hypernyms(swim, simulate_root=True))
print()

# Path and Wu-Palmer return a value only through the fake root, which has no meaning.
print(f"Path              {man.path_similarity(swim):.4f}    without fake root: {man.path_similarity(swim, simulate_root=False)}")
print(f"Wu-Palmer         {man.wup_similarity(swim):.4f}    without fake root: {man.wup_similarity(swim, simulate_root=False)}")

# Leacock-Chodorow and Lin raise an error with a noun and a verb.
try:
    man.lch_similarity(swim)
except nltk.corpus.reader.wordnet.WordNetError as error:
    print(f"Leacock-Chodorow  error: {error}")
try:
    man.lin_similarity(swim, brown_ic)
except nltk.corpus.reader.wordnet.WordNetError as error:
    print(f"Lin               error: {error}")

Roots: [Synset('entity.n.01')] [Synset('travel.v.01')]
LCS: []
LCS with fake root: [Synset('*ROOT*')]

Path              0.1000    without fake root: None
Wu-Palmer         0.1818    without fake root: None
Leacock-Chodorow  error: Computing the lch similarity requires Synset('man.n.01') and Synset('swim.v.01') to have the same part of speech.
Lin               error: Computing the least common subsumer requires Synset('man.n.01') and Synset('swim.v.01') to have the same part of speech.


## Adjectives and adverbs

The statement has no adjectives or adverbs, but the same check shows why the four similarities are only used for nouns and verbs. Adjectives and adverbs have no hypernym hierarchy in WordNet.

In [4]:
# Adjectives and adverbs have no hypernyms, so two of them only share the fake root.
# Most frequent synset of each word, as for the pairs of the statement.
for w1, w2, pos in [("good", "bad", wn.ADJ), ("quickly", "slowly", wn.ADV)]:
    s1, s2 = wn.synsets(w1, pos)[0], wn.synsets(w2, pos)[0]
    print(f"{s1.name()} - {s2.name()}")
    print("  Hypernyms:", s1.hypernyms(), s2.hypernyms())
    print("  LCS with fake root:", s1.lowest_common_hypernyms(s2, simulate_root=True))
    print(f"  Path {s1.path_similarity(s2):.4f}    Leacock-Chodorow {s1.lch_similarity(s2):.4f}    Wu-Palmer {s1.wup_similarity(s2):.4f}")
    # The Brown information content only has nouns and verbs.
    try:
        s1.lin_similarity(s2, brown_ic)
    except nltk.corpus.reader.wordnet.WordNetError as error:
        print(f"  Lin error: {error}")

good.a.01 - bad.a.01
  Hypernyms: [] []
  LCS with fake root: [Synset('*ROOT*')]


  Path 0.3333    Leacock-Chodorow -0.4055    Wu-Palmer 0.5000
  Lin error: Information content file has no entries for part-of-speech: a
quickly.r.01 - slowly.r.01
  Hypernyms: [] []
  LCS with fake root: [Synset('*ROOT*')]
  Path 0.3333    Leacock-Chodorow -0.4055    Wu-Palmer 0.5000
  Lin error: Information content file has no entries for part-of-speech: r


## Leacock-Chodorow normalization

Compute the maximum Leacock-Chodorow of each category (a synset compared with itself), used to scale Leacock-Chodorow to [0, 1].

In [5]:
# Path, Wu-Palmer and Lin already are between 0 and 1.
# Leacock-Chodorow = -log(p / 2D), where p is the path length and D the depth of the taxonomy,
# so it is not in [0, 1]. Its maximum is reached when both synsets are the same (p = 1): log(2D), 
# which depends on the category. Leacock-Chodorow is normalized by dividing it by this maximum.
lch_max = {pos: s.lch_similarity(s) for pos, s in [(wn.NOUN, synsets["man"]), (wn.VERB, synsets["swim"])]}
for pos, value in lch_max.items():
    print(f"Maximum Leacock-Chodorow for category '{pos}': {value:.4f}")

Maximum Leacock-Chodorow for category 'n': 3.6376
Maximum Leacock-Chodorow for category 'v': 3.2581


## Similarities of all pairs

LCS and the four similarities of every pair of words of the same category.

In [6]:
# LCS and similarities of every pair of words of the same category (15 noun pairs, 3 verb pairs).
rows = []
for w1, w2 in combinations(synsets, 2):
    s1, s2 = synsets[w1], synsets[w2]
    if s1.pos() != s2.pos():
        continue
    rows.append({
        "pair": f"{w1}-{w2}",
        "pos": s1.pos(),
        "LCS": ", ".join(s.name() for s in s1.lowest_common_hypernyms(s2)),
        "Path": s1.path_similarity(s2),
        "LC": s1.lch_similarity(s2),
        "LC norm": s1.lch_similarity(s2) / lch_max[s1.pos()],
        "Wu-Palmer": s1.wup_similarity(s2),
        "Lin": s1.lin_similarity(s2, brown_ic),
    })

similarities = pd.DataFrame(rows).set_index("pair")
measures = ["Path", "LC norm", "Wu-Palmer", "Lin"]
similarities.round(4)

,pos,LCS,Path,LC,LC norm,Wu-Palmer,Lin
pair,,,,,,,
man-girl,n,adult.n.01,0.2500,2.2513,0.6189,0.6316,0.7135
man-boy,n,male.n.02,0.3333,2.5390,0.6980,0.6667,0.7295
man-woman,n,adult.n.01,0.3333,2.5390,0.6980,0.6667,0.7871
man-dog,n,organism.n.01,0.1429,1.6917,0.4651,0.6667,0.2922
man-cat,n,organism.n.01,0.0833,1.1527,0.3169,0.5217,0.2915
swim-walk,v,travel.v.01,0.3333,2.1595,0.6628,0.3333,0.4910
swim-run,v,travel.v.01,0.2500,1.8718,0.5745,0.2857,0.4650
girl-boy,n,person.n.01,0.1667,1.8458,0.5074,0.6316,0.2927
girl-woman,n,woman.n.01,0.5000,2.9444,0.8094,0.6316,0.9068


## Ranking of the noun pairs

The four most similar noun pairs according to each measure, and the rank of every noun pair (1 = most similar, equal values share the same rank). Comparing the columns shows which measures order the pairs in the same way.

In [7]:
nouns = similarities[similarities["pos"] == wn.NOUN]

# Most similar pairs according to each measure.
for measure in measures:
    top = nouns[measure].sort_values(ascending=False).head(4)
    print(f"{measure:<10}", ", ".join(f"{pair} ({value:.3f})" for pair, value in top.items()))

# Do the measures rank the pairs in the same way? Rank of each noun pair by each measure.
nouns[measures].rank(ascending=False, method="min").astype(int).sort_values("Path")

Path       girl-woman (0.500), man-boy (0.333), man-woman (0.333), man-girl (0.250)
LC norm    girl-woman (0.809), man-boy (0.698), man-woman (0.698), man-girl (0.619)
Wu-Palmer  dog-cat (0.857), man-boy (0.667), man-woman (0.667), man-dog (0.667)
Lin        girl-woman (0.907), dog-cat (0.877), man-woman (0.787), man-boy (0.729)


,Path,LC norm,Wu-Palmer,Lin
pair,,,,
girl-woman,1,1,8,1
man-boy,2,2,2,4
man-woman,2,2,2,3
man-girl,4,4,8,5
boy-woman,5,5,2,6
dog-cat,5,5,1,2
girl-boy,7,7,8,7
man-dog,8,8,2,8
boy-dog,8,8,2,14


## Same group vs. person-animal

Compare the pairs of two people or two animals with the person-animal pairs. A good measure should separate the two groups.

In [8]:
# Nouns: pairs of two people or of two animals should be more similar than a person and an animal.
# A person-animal pair contains dog or cat, but is not dog-cat.
mixed = nouns.index.str.contains("dog|cat") & (nouns.index != "dog-cat")
same_group = nouns[~mixed][measures]
mixed_group = nouns[mixed][measures]
pd.DataFrame({
    "same group: min": same_group.min(),
    "same group: mean": same_group.mean(),
    "person-animal: max": mixed_group.max(),
    "person-animal: mean": mixed_group.mean(),
}).round(4)

,same group: min,same group: mean,person-animal: max,person-animal: mean
Path,0.1667,0.2833,0.1429,0.1101
LC norm,0.5074,0.6353,0.4651,0.3836
Wu-Palmer,0.6316,0.6788,0.6667,0.5871
Lin,0.2927,0.6607,0.2922,0.2767


## Ranking of the verb pairs

Ranking of the three verb pairs by each measure, and the hypernym chain of each verb to explain it.

In [9]:
# Verbs: the three pairs sorted by each measure, and the hypernym chain of each verb.
verbs = similarities[similarities["pos"] == wn.VERB]
for measure in measures:
    ranking = verbs[measure].sort_values(ascending=False)
    print(f"{measure:<10}", " > ".join(f"{pair} ({value:.3f})" for pair, value in ranking.items()))

print()
for verb in ["swim", "walk", "run"]:
    chain = [synsets[verb]] + list(synsets[verb].closure(lambda s: s.hypernyms()))
    print(f"{verb:<5}", " -> ".join(s.name() for s in chain), "|", synsets[verb].definition())

Path       swim-walk (0.333) > swim-run (0.250) > walk-run (0.250)
LC norm    swim-walk (0.663) > swim-run (0.575) > walk-run (0.575)
Wu-Palmer  swim-walk (0.333) > swim-run (0.286) > walk-run (0.286)
Lin        walk-run (0.505) > swim-walk (0.491) > swim-run (0.465)

swim  swim.v.01 -> travel.v.01 | travel through water
walk  walk.v.01 -> travel.v.01 | use one's feet to advance; advance by steps
run   run.v.01 -> travel_rapidly.v.01 -> travel.v.01 | move fast by using one's feet, with one foot off the ground at any given time


## Extra nouns

All the nouns of the statement are people or animals, so every pair shares at least `organism.n.01`. Two nouns from another domain (two vehicles) are added to see how the measures behave with less related pairs.

In [10]:
# Most frequent synset of each extra noun, compared with every noun.
extra = {word: wn.synsets(word, wn.NOUN)[0] for word in ["car", "bicycle"]}
all_nouns = {word: s for word, s in synsets.items() if s.pos() == wn.NOUN} | extra

rows = []
for w1, w2 in combinations(all_nouns, 2):
    if w1 not in extra and w2 not in extra:
        continue
    s1, s2 = all_nouns[w1], all_nouns[w2]
    rows.append({
        "pair": f"{w1}-{w2}",
        "LCS": ", ".join(s.name() for s in s1.lowest_common_hypernyms(s2)),
        "Path": s1.path_similarity(s2),
        "LC norm": s1.lch_similarity(s2) / lch_max[wn.NOUN],
        "Wu-Palmer": s1.wup_similarity(s2),
        "Lin": s1.lin_similarity(s2, brown_ic),
    })
extra_pairs = pd.DataFrame(rows).set_index("pair")

extra_pairs.sort_values("Lin", ascending=False).round(4)

,LCS,Path,LC norm,Wu-Palmer,Lin
pair,,,,,
car-bicycle,wheeled_vehicle.n.01,0.2000,0.5576,0.7273,0.7662
man-car,whole.n.02,0.0769,0.2949,0.4000,0.2218
woman-car,whole.n.02,0.0769,0.2949,0.4000,0.2212
girl-car,whole.n.02,0.0714,0.2745,0.3810,0.2024
man-bicycle,whole.n.02,0.0909,0.3408,0.4444,0.1980
woman-bicycle,whole.n.02,0.0909,0.3408,0.4444,0.1976
boy-car,whole.n.02,0.0769,0.2949,0.4000,0.1916
dog-car,whole.n.02,0.0769,0.2949,0.4000,0.1846
cat-car,whole.n.02,0.0556,0.2054,0.3200,0.1842


## Question

Only the 9 nouns and verbs have a synset. *the*, *with*, *and* and *whilst* are not in WordNet, and *a* and *after* are only there as other categories. For all 9 words the first sense is the most frequent one.

Similarities are only computed between words of the same category (15 noun pairs and 3 verb pairs). Between a noun and a verb there is no LCS: Leacock-Chodorow and Lin raise an error, and Path and Wu-Palmer only return a value through a fake common root. Adjectives and adverbs are not compared either: they have no hypernym hierarchy, so their values only come from the fake root, and Lin has no information content for them.

Path, Wu-Palmer and Lin are already between 0 and 1. Leacock-Chodorow is not: its maximum is log(2D), which is 3.6376 for nouns and 3.2581 for verbs. It is divided by this maximum, so that it is in [0, 1] and nouns and verbs can be compared.

### What similarity seems better?

**Lin similarity.** The reasons are:

- **It separates the related pairs from the unrelated ones the most.** The pairs of two people or two animals have a mean Lin of 0.6607 and the person-animal pairs 0.2767, the largest difference of the four measures. With the extra nouns, car-bicycle gets 0.7662 and all the pairs of a person or an animal with a vehicle (at most 0.2218) are below every pair of the statement.
- **Its top pairs are the expected ones.** girl-woman (0.907), dog-cat (0.877), man-woman (0.787) and man-boy (0.729).
- **It gives the most sensible order for the verbs.** It is the only measure that puts walk-run first and their definitions are close ("use one's feet to advance" / "move fast by using one's feet"). The other three measures put swim-walk first only because `run.v.01` has one more level (`travel_rapidly.v.01`) under `travel.v.01`.

Lin also has limits. It needs an information content computed on a corpus (Brown here), and it depends on the synset chosen: girl-boy (0.2927) and boy-woman (0.3184) are just above the person-animal pairs, because the most frequent synset of *girl* is "a young woman" and the one of *boy* is "a youthful male person", so their LCS is only the general `person.n.01`.

About the other measures:

- **Path** only counts edges, so it depends on how deep each word is in the taxonomy. dog-cat (0.200) gets less than man-girl (0.250), because dog and cat are deep in the animal taxonomy. With the extra nouns, man-bicycle (0.0909) is even more similar than man-cat (0.0833).
- **Leacock-Chodorow** is a logarithm of the path length, so for words of the same category it ranks the pairs exactly as Path (the two columns of the ranking are identical). The normalization only changes the scale.
- **Wu-Palmer** gives the same value (0.6667) to man-dog, man-woman and man-boy, and its ranking is the most different from the others: girl-woman, first for the other three measures, is only 8th. Its values are compressed between 0.50 and 0.86 for the nouns of the statement, but only because they are all organisms: with the extra nouns they go down to 0.32 (cat-car).